# DeBERTa-v3-small (PRETRAINED TRANSFORMER)

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


CompletedProcess(args=['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], returncode=0)

In [2]:
import os, warnings, pickle
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

INPUT_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
OUTPUT_DIR = Path('/kaggle/working')

Device : cuda
GPU    : Tesla T4


In [4]:
# creating logs and models folder in kaggle 

OUTPUT_DIR = Path('/kaggle/working')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "logs").mkdir(
    parents=True,
    exist_ok=True
)

(OUTPUT_DIR / "models").mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
# config define randomly ...

CFG = dict(
    model_name   = 'microsoft/deberta-v3-small',
    max_len      = 256,    # tokens for (prompt + ONE option)
    batch_size   = 16,    
    lr           = 2e-5,   # transformer learning rate
    weight_decay = 0.01,
    epochs       = 5,      # transformers converge fast
    warmup_ratio = 0.1,    # 10% of steps for warmup
    patience     = 3,
    seed         = 42,
)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('Config:')
for k, v in CFG.items(): print(f'  {k:<14}: {v}')

Config:
  model_name    : microsoft/deberta-v3-small
  max_len       : 256
  batch_size    : 16
  lr            : 2e-05
  weight_decay  : 0.01
  epochs        : 5
  warmup_ratio  : 0.1
  patience      : 3
  seed          : 42


In [6]:
train_df = pd.read_csv(INPUT_DIR / 'train.csv')
test_df  = pd.read_csv(INPUT_DIR / 'test.csv')

# Lowercase it 
for col in ['prompt','A','B','C','D','E']:
    train_df[col] = train_df[col].str.lower().str.strip()
    test_df[col]  = test_df[col].str.lower().str.strip()

In [7]:
np.random.seed(CFG['seed'])
tr_idx, va_idx = [], []
for ans in 'ABCDE':
    idx = train_df[train_df['answer']==ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx)*0.8)
    tr_idx += idx[:cut]
    va_idx += idx[cut:]

tr_df = train_df.loc[tr_idx].reset_index(drop=True)
va_df = train_df.loc[va_idx].reset_index(drop=True)
te_df = test_df.copy()

In [8]:
A2I = {'A':0,'B':1,'C':2,'D':3,'E':4}
I2A = {v:k for k,v in A2I.items()}

print(f'Train:{len(tr_df)} | Val:{len(va_df)} | Test:{len(te_df)}')

Train:1599 | Val:401 | Test:500
